# Colab fallback — Distributed Systems / Big Data

Use this **only** if the Docker image will not run on your machine. Tell me if you
are using it, because your numbers are not directly comparable to the rest of the class.

## What works here
Spark, DataFrames, shuffles, `explain()`, Parquet, DuckDB, the whole S8/S9 pipeline.

## What does NOT work here, honestly
| Thing | Why | What you do instead |
|---|---|---|
| **Spark UI (S7)** | Colab has no reachable port | `stage_report()` in Step 5 — same numbers, text form |
| **S1 speedup (TODO 2)** | Colab gives 2 vCPU, the class gets 4 | Report your core count; expect ~2× not ~4× |
| **Session persistence** | Colab disconnects after ~90 min idle | Re-run Steps 1–2; the data rebuild is the slow part |
| **Docker's 4 GB cap** | Colab has ~13 GB | Step 3 caps memory in a child process instead |

Run the steps in order.

In [ ]:

#@title Step 1 — environment check and install  { display-mode: "form" }
import sys, subprocess, platform, os, textwrap

PY = sys.version_info
print(f"Colab Python: {platform.python_version()}")

# PySpark 3.5.x does not support Python 3.12. Colab moves its runtime without
# warning, so pick the version that will actually import.
if (PY.major, PY.minor) <= (3, 11):
    SPARK_PIN, SPARK_MAJOR = "pyspark==3.5.9", 3
else:
    SPARK_PIN, SPARK_MAJOR = "pyspark==4.0.1", 4
    print(textwrap.dedent("""
    ############################################################
    #  WARNING — READ THIS AND TELL YOUR TEACHER
    #  Colab is on Python 3.12+, so this notebook installed
    #  Spark 4.x instead of the 3.5.9 the class uses.
    #  explain() output and some UI/metric names DIFFER.
    #  Your answers will not match the debrief word for word.
    ############################################################"""))

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                SPARK_PIN, "duckdb==1.1.3", "pyarrow==16.1.0"], check=True)

import pyspark
print("PySpark:", pyspark.__version__)
print("Java   :", subprocess.run(["java","-version"],capture_output=True,text=True).stderr.splitlines()[0])
print("CPUs   :", os.cpu_count(), " (the Docker class gets 4 — note yours)")
import psutil; print("RAM    :", f"{psutil.virtual_memory().total/1024**3:.1f} GB")


In [ ]:

#@title Step 2 — build the TP datasets  { display-mode: "form" }
# Same generator the Docker image uses, so the data is identical.
REPO_RAW = "https://raw.githubusercontent.com/REPLACE-ME/bigdata-course/main"  #@param {type:"string"}
PROFILE  = "small"  #@param ["tiny", "small", "full"]

import os, subprocess, sys, urllib.request, pathlib
os.makedirs("/content/course", exist_ok=True)

ok = True
for f in ("prepare_tp_data.py", "spark_evidence.py"):
    try:
        urllib.request.urlretrieve(f"{REPO_RAW}/scripts/{f}", f"/content/course/{f}")
        print("fetched", f)
    except Exception as e:
        ok = False
        print(f"COULD NOT FETCH {f}: {e}")
if not ok:
    print("\n-> Upload scripts/prepare_tp_data.py and scripts/spark_evidence.py "
          "manually with the file panel on the left, into /content/course/")

sys.path.insert(0, "/content/course")

# Real data if reachable, synthetic (schema-identical) if not.
SEED_URL  = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet"
ZONES_URL = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"
args = ["--out", "/content/data", "--profile", PROFILE]
try:
    urllib.request.urlretrieve(ZONES_URL, "/content/zones.csv")
    urllib.request.urlretrieve(SEED_URL, "/content/seed.parquet")
    args += ["--seed-file", "/content/seed.parquet", "--zones-file", "/content/zones.csv"]
    print("using the real TLC data")
except Exception as e:
    args += ["--synthetic"]
    print("TLC unreachable -> synthetic seed:", e)

subprocess.run([sys.executable, "/content/course/prepare_tp_data.py"] + args, check=True)


In [ ]:

#@title Step 3 — the memory wall (this is what replaces the container's 4 GB limit)
# Colab gives you ~13 GB. The class container is capped at 4 GB. Without a cap your
# S1 measurements are meaningless, because the file that crushes everyone else fits
# comfortably for you.
#
# run_capped() runs code in a child process with a hard address-space limit, so you
# get a clean MemoryError at a KNOWN ceiling instead of a Colab crash.
import subprocess, sys, textwrap, time

def run_capped(code: str, gb: float = 4.0, timeout: int = 900):
    prog = textwrap.dedent(f"""
        import resource, sys, time
        resource.setrlimit(resource.RLIMIT_AS, ({int(gb*1024**3)}, {int(gb*1024**3)}))
        t0 = time.time()
        try:
{textwrap.indent(textwrap.dedent(code), ' ' * 12)}
            print(f"__ELAPSED__ {{time.time()-t0:.2f}}")
        except MemoryError:
            print(f"__MEMORYERROR__ after {{time.time()-t0:.2f}}s "
                  f"— this is the single-machine wall, at a {gb} GB ceiling")
            sys.exit(0)
    """)
    r = subprocess.run([sys.executable, "-c", prog], capture_output=True,
                       text=True, timeout=timeout)
    print(r.stdout or r.stderr[-2000:])
    return r

# demo: the S1 TODO 1 shape
run_capped("""
import pandas as pd
df = pd.read_csv("/content/data/s1/trips_big.csv")
print("rows:", len(df), "memory:", df.memory_usage(deep=True).sum()/1024**3, "GB")
""", gb=4.0)


In [ ]:

#@title Step 4 — Spark configured to match the class container
import os
os.makedirs("/content/spark-events", exist_ok=True)
os.environ["SPARK_EVENT_DIR"] = "/content/spark-events"

from pyspark.sql import SparkSession
spark = (SparkSession.builder
    .master("local[4]")                                  # match the container, even on 2 vCPU
    .appName("course")
    .config("spark.driver.memory", "3g")
    .config("spark.sql.shuffle.partitions", "16")        # 200 is unreadable on a laptop
    .config("spark.sql.adaptive.enabled", "false")       # ON DELIBERATELY IN S7, NOT BEFORE
    .config("spark.sql.adaptive.skewJoin.enabled", "false")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///content/spark-events")
    .getOrCreate())

print("Spark", spark.version, "| cores", spark.sparkContext.defaultParallelism)
print("NOTE: there is NO Spark UI on Colab. Use stage_report() in the next cell.")


In [ ]:

#@title Step 5 — the Spark UI replacement (this is how you do S7 on Colab)
import sys; sys.path.insert(0, "/content/course")
from spark_evidence import stage_report, skew_report, before_after

import time
t0 = time.time()
trips = spark.read.parquet("/content/data/s7/trips_skewed")
zones = spark.read.parquet("/content/data/s7/zones.parquet")
(trips.join(zones.hint("shuffle_hash"), trips.PULocationID == zones.LocationID)
      .groupBy("pickup_borough").count().collect())
elapsed = time.time() - t0

spark.sparkContext._jsc.sc().listenerBus().waitUntilEmpty(10_000)  # flush the event log
stage_report()          # tasks, median vs max, skew ratio, shuffle bytes
# skew_report(stage_id=2)          # histogram of one stage's task durations
# before_after("baseline", elapsed) # build the 4-row project table


In [ ]:

#@title Step 6 — final check: screenshot this output
import json, platform, os
m = json.load(open("/content/data/MANIFEST.json"))
name = "PUT YOUR NAME HERE"  #@param {type:"string"}

checks = {
    "pyspark":     __import__("pyspark").__version__,
    "python":      platform.python_version(),
    "cpus":        os.cpu_count(),
    "data profile": m["profile"],
    "columns":     m["curated_columns"],
    "dominant key": f'{m["dominant_key"]} {m["dominant_key_share"]:.1%}',
    "fingerprint": m["fingerprint"],
}
bad = m["dominant_key_share"] < 0.5 or m["curated_columns"] < 30
print("=" * 58)
print(f"  COLAB FALLBACK — {name}")
for k, v in checks.items():
    print(f"  {k:<14}: {v}")
print(f"  status        : {'PROBLEM - tell the teacher' if bad else 'READY'}")
print("=" * 58)
print("\nSend this screenshot AND tell me you are on the Colab fallback,")
print("so I know your timings sit in a different column of the shared sheet.")


## If something breaks

1. `Runtime -> Disconnect and delete runtime`, then run every cell from the top.
   Colab keeps half-installed packages around and they poison everything after them.
2. If `prepare_tp_data.py` fails to download, upload it by hand into `/content/course/`.
3. If you hit the Colab disk limit, set `PROFILE = "tiny"` in Step 2.
4. Still stuck: post in the debug channel with the output of Step 6, not a description
   of the error.